In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
from pathlib import Path

In [2]:
from sklearn import metrics
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
feature_selected = []
feature_name = 'FM'
for k in range(0, 64):

    formatted = str(k).zfill(2)
    feature_selected.append('A' + formatted)

In [5]:
df_all = pd.read_csv("US_FM_Corn_Data_shared.csv")

In [6]:
print(df_all.shape)

(6325, 106)


# Evaluation

In [7]:
random_seed = 20

list_year = []
list_num = []

list_RMSE_RF = []
list_R2_RF = []

list_RMSE_XGB = []
list_R2_XGB = []

list_RF_models = []

## Spatial-Temporal CV

In [8]:
list_year.append('Spatial-Temporal CV')
list_num.append(df_all.shape[0])

In [9]:
rf_model  = RandomForestRegressor(n_estimators=200)
XGB_model = GradientBoostingRegressor(n_estimators=200)

In [11]:
r2_scores = []
rmse_scores = []

for i in range(5):
    # Create unique combinations of FIPS and year
    unique_keys = df_all[['State', 'year']].drop_duplicates()

    # Split these unique combinations into train/test
    train_keys, test_keys = train_test_split(unique_keys, test_size=0.3, random_state=i)

    # Merge back to get full train/test DataFrames
    df_train = df_all.merge(train_keys, on=['State', 'year'])
    df_test = df_all.merge(test_keys, on=['State', 'year'])

    X_train = df_train[feature_selected]
    y_train = df_train['yield'] * 0.0673

    X_test = df_test[feature_selected]
    y_test = df_test['yield'] * 0.0673
    
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    # R²
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    
    # RMSE
    rmse = mean_squared_error(y_test, y_pred)
    rmse_scores.append(rmse)

In [ ]:
R2_RF = np.mean(r2_scores)
RMSE_RF = np.mean(rmse_scores)
print(random_seed)
print('RF')
print(R2_RF, RMSE_RF)

list_RMSE_RF.append(RMSE_RF)
list_R2_RF.append(R2_RF)

20
RF
0.7754200294093325 1.1879362674097649


In [15]:
r2_scores = []
rmse_scores = []

for i in range(5):
    # Create unique combinations of FIPS and year
    unique_keys = df_all[['State', 'year']].drop_duplicates()

    # Split these unique combinations into train/test
    train_keys, test_keys = train_test_split(unique_keys, test_size=0.3, random_state=i)

    # Merge back to get full train/test DataFrames
    df_train = df_all.merge(train_keys, on=['State', 'year'])
    df_test = df_all.merge(test_keys, on=['State', 'year'])

    X_train = df_train[feature_selected]
    y_train = df_train['yield'] * 0.0673

    X_test = df_test[feature_selected]
    y_test = df_test['yield'] * 0.0673
    
    XGB_model.fit(X_train, y_train)
    y_pred = XGB_model.predict(X_test)
    
    # R²
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    
    # RMSE
    rmse = mean_squared_error(y_test, y_pred)
    rmse_scores.append(rmse)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [16]:
R2_XGB = np.mean(r2_scores)
RMSE_XGB = np.mean(rmse_scores)
print(random_seed)
print('XGB')
print(R2_XGB, RMSE_XGB)

list_RMSE_XGB.append(RMSE_XGB)
list_R2_XGB.append(R2_XGB)

20
XGB
0.7830551581323796 1.166085582086429


# Yearly CV

In [16]:
for i, year in enumerate(years):
    
    df_train = df_all[df_all['year'] != year]
    df_test  = df_all[df_all['year'] == year]

    X_train = df_train[feature_selected]
    y_train = df_train['yield'] * 0.0673

    X_test = df_test[feature_selected]
    y_test = df_test['yield']  * 0.0673

    rf_model = RandomForestRegressor(n_estimators=200)

    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)
    R2 = r2_score(y_test, y_pred)
    RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
    print(year)
    print(X_test.shape)
    print(R2, RMSE)

    if i == 0:
        y_test_all = y_test
        y_pred_all = y_pred
    else:
        y_test_all = np.concatenate((y_test_all, y_test), axis=0)
        y_pred_all = np.concatenate((y_pred_all, y_pred), axis=0)

    df_test.loc[:, 'pred'] = y_pred

    list_year.append(year)
    list_num.append(df_test.shape[0])
    list_RMSE_RF.append(RMSE)
    list_R2_RF.append(R2)

    list_RF_models.append(rf_model)



2017
(829, 64)
0.83149385277128 0.9323195664444669


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_73776/677010377.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2018
(734, 64)
0.6925365640008061 1.3432321981432565


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_73776/677010377.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2019
(679, 64)
0.733192913441017 0.9986586247802891


...

/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_73776/677010377.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2023
(808, 64)
0.8017783962548026 1.1125737758238612


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_73776/677010377.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2024
(698, 64)
0.7263093810397261 1.2428251397926682


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_73776/677010377.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


In [18]:
print("RF all")
R2 = r2_score(y_test_all, y_pred_all)
RMSE = np.sqrt(mean_squared_error(y_test_all, y_pred_all))

list_year.append('All')
list_num.append(y_test_all.shape[0])
list_RMSE_RF.append(RMSE)
list_R2_RF.append(R2)

print(r2_score(y_test_all, y_pred_all), np.sqrt(mean_squared_error(y_test_all, y_pred_all)))

RF all
0.772897816368265 1.154450479173108


In [19]:
## XGB
for i, year in enumerate(years):
    
    df_train = df_all[df_all['year'] != year]
    df_test  = df_all[df_all['year'] == year]

    X_train = df_train[feature_selected]
    y_train = df_train['yield'] * 0.0673

    X_test = df_test[feature_selected]
    y_test = df_test['yield']  * 0.0673

    XGB_model = GradientBoostingRegressor(n_estimators=200)

    XGB_model.fit(X_train, y_train)
    y_pred = XGB_model.predict(X_test)
    R2 = r2_score(y_test, y_pred)
    RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
    print(year)
    print(X_test.shape)
    print(R2, RMSE)

    if i == 0:
        y_test_all = y_test
        y_pred_all = y_pred
    else:
        y_test_all = np.concatenate((y_test_all, y_test), axis=0)
        y_pred_all = np.concatenate((y_pred_all, y_pred), axis=0)

    df_test.loc[:, 'pred'] = y_pred

    list_RMSE_XGB.append(RMSE)
    list_R2_XGB.append(R2)

2017
(829, 64)
0.8202917452221461 0.9628107655553259


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_39268/158009915.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2018
(734, 64)
0.6966837094791732 1.3341425133312133


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_39268/158009915.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2019
(679, 64)
0.7375271085146351 0.9905139688784378


...

/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_39268/158009915.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2023
(808, 64)
0.7830316552826735 1.1639960704220418


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_39268/158009915.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


2024
(698, 64)
0.6860991452178286 1.3309946899356273


/var/folders/sc/lkhxdnzd1vx_6xhtx4xwm7hw0000gn/T/ipykernel_39268/158009915.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:, 'pred'] = y_pred


In [20]:
print("XGB all")
R2 = r2_score(y_test_all, y_pred_all)
RMSE = np.sqrt(mean_squared_error(y_test_all, y_pred_all))

list_RMSE_XGB.append(RMSE)
list_R2_XGB.append(R2)

print(r2_score(y_test_all, y_pred_all), np.sqrt(mean_squared_error(y_test_all, y_pred_all)))

XGB all
0.769744342530849 1.1624380257571343


In [21]:
df_eval = pd.DataFrame({
    'year': list_year,
    'num': list_num,
    'R2_RF': list_R2_RF,
    'RMSE_RF': list_RMSE_RF,
    'R2_XGB': list_R2_XGB,
    'RMSE_XGB': list_RMSE_XGB,
})
